# 07 · Model training và hyperparameter tuning

Đánh giá Random Forest, Extra Trees, KNN và HistGradientBoosting bằng outer 5-fold `StratifiedGroupKFold`. Tiền xử lý được fit trong pipeline trên outer train. HistGradientBoosting được tuning bằng `RandomizedSearchCV` với inner 2-fold stratified group CV riêng bên trong mỗi outer fold (nested CV), để outer validation fold không tham gia chọn tham số.

Metric chính: macro-F1; accuracy kèm theo. XGBoost, LightGBM và CatBoost không được cài trong môi trường hiện tại; HistGradientBoosting là lựa chọn boosting của scikit-learn.

In [ ]:
from pathlib import Path
from datetime import datetime
import sys, csv, platform, time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score
from sklearn.base import clone

SEED = 42
RUN_START = datetime.now()
ROOT = Path(r"C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2")
DATA_DIR = ROOT / "DATA_CHALLENGE2"
LOG_DIR = ROOT / "logs"
NB_DIR = ROOT / "notebooks"
sys.path.insert(0, str(NB_DIR))
from preprocessing import MusicPreprocessor

TRAIN_PATH = DATA_DIR / "train (1).csv"
TEST_PATH = DATA_DIR / "test (2).csv"
TARGET = "Class"
ID = "Id"
GROUP = "group_id"
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy dữ liệu raw: {TRAIN_PATH}")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = train[TARGET].astype(int)
X = train.drop(columns=[TARGET])
# Cùng nghệ sĩ + tên bài hát luôn thuộc một fold; không dùng nhãn để tạo group.
group_key = (train["Artist Name"].fillna("").astype(str).str.strip().str.lower()
             + "||" + train["Track Name"].fillna("").astype(str).str.strip().str.lower())
groups = pd.factorize(group_key)[0]
CV = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
SPLITS = list(CV.split(X, y, groups=groups))

def make_pipeline(model):
    # MusicPreprocessor chỉ fit trên từng phần train của fold.
    return Pipeline([("preprocess", MusicPreprocessor()), ("model", model)])

def append_run_log(stage, notebook, goal, steps, notes):
    end = datetime.now()
    elapsed = round((end - RUN_START).total_seconds(), 1)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    run_id = f"{stage}_{end:%Y%m%d_%H%M%S}"
    slug = "Tô-Doanh-Quang-Khu"
    md_path = LOG_DIR / f"{stage}_{end:%Y%m%d_%H%M%S}_{slug}.md"
    suffix = 2
    while md_path.exists():
        md_path = LOG_DIR / f"{stage}_{end:%Y%m%d_%H%M%S}_{slug}_{suffix}.md"
        suffix += 1
    record = {"run_id":run_id,"date":end.strftime("%Y-%m-%d"),"time":end.strftime("%H:%M:%S"),
        "runner":"Tô Doanh Quang Khu","stage":stage,"notebook":notebook,"goal":goal,
        "steps":steps,"duration_sec":elapsed,"python":sys.version.split()[0],
        "platform":platform.platform(),"notes":notes + f"; markdown_log={md_path.name}"}
    fields=["run_id","date","time","runner","stage","notebook","goal","steps","duration_sec","python","platform","notes"]
    exists=(LOG_DIR/"experiment_log.csv").exists() and (LOG_DIR/"experiment_log.csv").stat().st_size>0
    with (LOG_DIR/"experiment_log.csv").open("a",newline="",encoding="utf-8-sig") as f:
        writer=csv.DictWriter(f,fieldnames=fields)
        if not exists: writer.writeheader()
        writer.writerow(record)
    steps_md="\n".join(f"- {part.strip()}" for part in steps.split(";"))
    md=f"""# Nhật ký thí nghiệm – {stage}

| Mục | Giá trị |
|---|---|
| Người chạy | Ngô Thanh Phong |
| Ngày chạy | {end.strftime('%d/%m/%Y')} |
| Giờ bắt đầu – kết thúc | {RUN_START.strftime('%H:%M:%S')} – {end.strftime('%H:%M:%S')} ({elapsed} giây) |
| Notebook | `{notebook}` |
| Mục tiêu | {goal} |
| Môi trường | Python {sys.version.split()[0]} · {platform.platform()} |

## Các bước & kết quả

{steps_md}

## Nhận xét

{notes}
"""
    md_path.write_text(md,encoding="utf-8")
    print(f"Đã ghi CSV và Markdown: {md_path}")
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold

fixed_models={
    "RandomForest":make_pipeline(RandomForestClassifier(n_estimators=120,min_samples_leaf=2,class_weight="balanced_subsample",n_jobs=-1,random_state=SEED)),
    "ExtraTrees":make_pipeline(ExtraTreesClassifier(n_estimators=120,min_samples_leaf=2,class_weight="balanced",n_jobs=-1,random_state=SEED)),
    "KNN":make_pipeline(KNeighborsClassifier(n_neighbors=15,weights="distance",n_jobs=-1)),
}
param_distributions={
    "model__max_iter":[80,140],
    "model__learning_rate":[0.05,0.1],
    "model__max_leaf_nodes":[15,31],
    "model__l2_regularization":[0.0,1.0],
}
outer_rows=[]
search_rows=[]
for fold,(tr,va) in enumerate(SPLITS,start=1):
    assert set(groups[tr]).isdisjoint(set(groups[va]))
    for name,estimator in fixed_models.items():
        fitted=clone(estimator)
        fitted.fit(X.iloc[tr],y.iloc[tr])
        pred=fitted.predict(X.iloc[va])
        outer_rows.append({"model":name,"fold":fold,"macro_f1":f1_score(y.iloc[va],pred,average="macro"),"accuracy":accuracy_score(y.iloc[va],pred),"best_params":"fixed"})
    inner_cv=StratifiedGroupKFold(n_splits=2,shuffle=True,random_state=SEED+fold)
    search=RandomizedSearchCV(
        estimator=make_pipeline(HistGradientBoostingClassifier(random_state=SEED,max_iter=80)),
        param_distributions=param_distributions,n_iter=2,scoring="f1_macro",cv=inner_cv,
        random_state=SEED+fold,n_jobs=1,refit=True,return_train_score=False)
    search.fit(X.iloc[tr],y.iloc[tr],groups=groups[tr])
    tuned_pred=search.best_estimator_.predict(X.iloc[va])
    outer_rows.append({"model":"HistGradientBoosting tuned","fold":fold,
        "macro_f1":f1_score(y.iloc[va],tuned_pred,average="macro"),
        "accuracy":accuracy_score(y.iloc[va],tuned_pred),"best_params":str(search.best_params_)})
    search_rows.append({"outer_fold":fold,"inner_best_macro_f1":search.best_score_,"best_params":search.best_params_})
    print(f"Hoàn tất outer fold {fold}/5")

fold_results=pd.DataFrame(outer_rows)
summary=fold_results.groupby("model").agg(macro_f1_mean=("macro_f1","mean"),macro_f1_std=("macro_f1","std"),accuracy_mean=("accuracy","mean")).sort_values("macro_f1_mean",ascending=False)
display(summary.round(4))
display(pd.DataFrame(search_rows))
display(fold_results[["model","fold","macro_f1","accuracy"]].pivot(index="fold",columns="model",values="macro_f1").round(4))
best_name=summary.index[0]
print("Mô hình tốt nhất theo outer CV macro-F1:",best_name)

Hoàn tất outer fold 1/5


Hoàn tất outer fold 2/5


Hoàn tất outer fold 3/5


Hoàn tất outer fold 4/5


Hoàn tất outer fold 5/5


,macro_f1_mean,macro_f1_std,accuracy_mean
model,,,
HistGradientBoosting tuned,0.5864,0.0052,0.5542
RandomForest,0.5783,0.0093,0.5429
ExtraTrees,0.5493,0.0049,0.5033
KNN,0.4891,0.0097,0.4815


,outer_fold,inner_best_macro_f1,best_params
0,1,0.566965,"{'model__max_leaf_nodes': 15, 'model__max_iter..."
1,2,0.567093,"{'model__max_leaf_nodes': 31, 'model__max_iter..."
2,3,0.575831,"{'model__max_leaf_nodes': 15, 'model__max_iter..."
3,4,0.572989,"{'model__max_leaf_nodes': 15, 'model__max_iter..."
4,5,0.573059,"{'model__max_leaf_nodes': 15, 'model__max_iter..."


model,ExtraTrees,HistGradientBoosting tuned,KNN,RandomForest
fold,,,,
1,0.5551,0.5921,0.4876,0.5842
2,0.5476,0.5870,0.4954,0.5898
3,0.5538,0.5781,0.4915,0.5698
4,0.5460,0.5884,0.4979,0.5796
5,0.5440,0.5866,0.4731,0.5681


Mô hình tốt nhất theo outer CV macro-F1: HistGradientBoosting tuned


## Fit mô hình đã chọn trên toàn bộ train và dự đoán test

In [2]:
# Outer nested-CV scores select model family without test leakage. Refit tuning on all train for the final model.
if best_name == "HistGradientBoosting tuned":
    final_search=RandomizedSearchCV(
        estimator=make_pipeline(HistGradientBoostingClassifier(random_state=SEED,max_iter=80)),
        param_distributions=param_distributions,n_iter=2,scoring="f1_macro",cv=CV,
        random_state=SEED,n_jobs=1,refit=True,return_train_score=False)
    final_search.fit(X,y,groups=groups)
    final_model=final_search.best_estimator_
    final_params=final_search.best_params_
    print("Final tuning parameters:",final_params)
else:
    final_model=clone(fixed_models[best_name])
    final_model.fit(X,y)
test_pred=final_model.predict(test)
submission=pd.DataFrame({"Id":test[ID],"Class":test_pred})
out_path=ROOT/"outputs"/"07_main_models_predictions.csv"
out_path.parent.mkdir(parents=True,exist_ok=True)
submission.to_csv(out_path,index=False)
print("Đã lưu dự đoán không nhãn:",out_path,submission.shape)
print("Outer CV scores, not test scores.")

Final tuning parameters: {'model__max_leaf_nodes': 31, 'model__max_iter': 80, 'model__learning_rate': 0.05, 'model__l2_regularization': 0.0}


Đã lưu dự đoán không nhãn: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2\outputs\07_main_models_predictions.csv (3600, 2)
Outer CV scores, not test scores.


## Ghi nhật ký kết quả

In [3]:
summary_text="; ".join(f"{name}: macro-F1={r.macro_f1_mean:.6f} ± {r.macro_f1_std:.6f}, accuracy={r.accuracy_mean:.6f}" for name,r in summary.iterrows())
param_text="; ".join(f"outer fold {r['outer_fold']}: {r['best_params']}" for r in search_rows)
append_run_log("07_Main_Models","07_Main_Models.ipynb","So sánh mô hình và nested hyperparameter tuning theo stratified group CV",
    f"Outer 5-fold StratifiedGroupKFold; inner 3-fold tuning; raw train={train.shape}; groups={pd.Series(groups).nunique()}; {summary_text}; tuning={param_text}; prediction_file={out_path}",
    f"Chọn {best_name} theo outer macro-F1; preprocessing fit trong từng fold; test không nhãn, không chấm điểm")

Đã ghi CSV và Markdown: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2\logs\07_Main_Models_20261112_192220_Tô-Doanh-Quang-Khu.md
